# Mini-Project 3

## <span style="color: blue;">Step 1: Dataset (Creation and Preprocessing)</span>

**Dataset:** Memes Images: OCR Data (Kaggle). This dataset has two components: 
- **Text:**
- **Images:** 16 sample PNG files (8 COVID memes, 8 US political memes) included as actual image files (.png)

Notably, these two meme categories let us ask a comparative question. Namely, does the same internet meme format function differently when applied to electorial politics vs. a public health crisis? Moreover, the entity tags are what make this dataset interesting, i.e., instead of just asking 'is this meme positive or negative,' we can ask 'who exactly is getting blamed, and who is getting credit.

### 1.1: Load Libraries and Data

In [ ]:
# imports 
import pandas as pd 
import numpy as np 
import ast 
import os 
import zipfile
from collections import Counter 

# for text analysis 
import nltk
from nltk.corpus import stopwords 
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('stopwords', quiet=True)
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt', quiet=True)
from sklearn.feature_extraction.text import TfidfVectorizer

# for image analysis 
from PIL import Image
import cv2

# for visualizations 
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from wordcloud import WordCloud, STOPWORDS

# sanity check line 
print("All libraries loaded successfully")

All libraries loaded successfully


In [ ]:
# load the data 
train_df = pd.read_csv('data/text_with_ocr.csv')
val_df = pd.read_csv('data/validation_data.csv')

# quick look at the data 
print(f"Training rows: {len(train_df)}")
print(f"Validation rows: {len(val_df)}")
print(f"Columns: {list(train_df.columns)}")
train_df.head(3)

### 1.2: Inspect and Understand the Structure

In [ ]:
# what do the entity tags look like 
    # note that tags are stored as string representations of Python lists, e.g. "['donald trump']"
# so they must be parsed properly 
def parse_tag(val):
    """ Safely parse a tag column from string-list to actual list"""
    try: 
        result = ast.literal_eval(val)
        if isinstance(result, list):
            return result 
        return []
    except: 
        return []
    
# apply the parser to all entity columns 
for col in ['hero', 'villain', 'victim', 'other']:
    train_df[col + '_parsed'] = train_df[col].apply(parse_tag)
    val_df[col + '_parsed'] = val_df[col].apply(parse_tag)

# add a category column based on the image filename prefix 
    # i.e., images named 'covid_memes_X.png' are COVID; 'memes_X.png' are US politics
train_df['category'] = train_df['image'].apply(
    lambda x: 'COVID' if str(x).startswith('covid') else 'US Politics'
)
val_df['category'] = val_df['image'].apply(
    lambda x: 'COVID' if str(x).startswith('covid') else 'US Politics'
)

print("Category breakdown (train):") 
print(train_df['category'].value_counts())
print()
print("Sample row OCR text:")
print(train_df['OCR'].iloc[0])

### 1.3: Preprocessing the OCR Text

In [ ]:
# clean OCR text 
    # OCR from memes is inherently noisy 
        # (e.g., picks up watermarks, usernames, website URLs, and formatting artifacts) 
# Here: do light cleaning (but careful not to string too much because the meme language is intentional)

import re 
stop_words = set(stopwords.words('english'))

def clean_ocr(text):
    """
    Light cleaning for meme OCR text: 
    - lowercase 
    - remove URLs and handles 
    - remove special characters but keep spaces 
    - strip extra whitespace 
    """
    if not isinstance(text, str):
        return ''
    text = text.lower()    # lowercase
    # remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text) 
    # remove @handles
    text = re.sub(r'@\w+', '', text) 
    # keep only letters + spaces 
    text = re.sub(r'[^a-z\s]', ' ', text) 
    # collapse whitespace 
    text = re.sub(r'\s+', ' ', text).strip() 

    return text 

train_df['ocr_clean'] = train_df['OCR'].apply(clean_ocr)
val_df['ocr_clean'] = val_df['OCR'].apply(clean_ocr)

# also compute word count on cleaned text (useful for EDA)
train_df['word_count'] = train_df['ocr_clean'].apply(lambda x: len(x.split()))

print("Preprocessing complete.")
print(f"Average word count per meme: {train_df['word_count'].mean():.1f}")
print(f"Median word count: {train_df['word_count'].median():.1f}")